<a href="https://colab.research.google.com/github/jarekwan/PROJEKT_SCANNER/blob/main/modul_pobieranie.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
os.makedirs('/content/drive/MyDrive/projekt_test', exist_ok=True)

print("folder ready")

Mounted at /content/drive
folder ready


In [12]:
%%writefile /content/drive/MyDrive/projekt_test/modul_pobieranie.py
# -*- coding: utf-8 -*-

import json
from pathlib import Path

import pandas as pd
import requests


def run():
    tik = input("podaj ticker dla danych z api twelve data: ").strip().upper()

    fd = Path("/content/drive/MyDrive/projekt_test")

    kl = "0ae01ea2dc5a43b58a1514eebfdf7ee8"
    url = "https://api.twelvedata.com/time_series"
    par = {
        "symbol": tik,
        "interval": "1day",
        "outputsize": 5000,
        "apikey": kl
    }

    r = requests.get(url, params=par, timeout=30)
    r.raise_for_status()
    d = r.json()

    fp = fd / f"{tik}_twelve.json"
    with open(fp, "w", encoding="utf-8") as f:
        json.dump(d, f, ensure_ascii=False, indent=2)

    print("glowne klucze odpowiedzi:", list(d.keys()))

    if d.get("status") == "error":
        print("ticker nie istnieje albo twelve data go nie rozpoznaje")
        print(d.get("message", "nieznany blad"))
        return

    if "values" not in d:
        print("brak sekcji values w odpowiedzi twelve data")
        return

    ls = d["values"]

    if not ls:
        print("ticker istnieje, ale brak danych dziennych")
        return

    print("klucze w rekordzie:", list(ls[0].keys()))

    df = pd.DataFrame(ls)

    print("kolumny przed zmiana nazw:", list(df.columns))

    df = df.rename(columns={"datetime": "data"})

    ocz = ["data", "open", "high", "low", "close", "volume"]
    brak = [k for k in ocz if k not in df.columns]

    if brak:
        print("brakuje kolumn:", brak)
        print("rzeczywiste kolumny po zmianie nazw:", list(df.columns))
        return

    df = df[ocz]
    print("zapisano:", fp)
    display(df.head(10))

Overwriting /content/drive/MyDrive/projekt_test/modul_pobieranie.py
